# 20.4 大底座裡，這輪究竟是哪幾張配方表在學？


接手一本已有許多內容的筆記，可以在旁邊加修訂便條，不一定要把每一頁重寫。LoRA就是一種在原有矩陣運算旁加入小修正的方法。本章仍用示範回答教模型，但開放更新的是指定位置的修正分支。要說「真的微調過」，至少要核對分支收到梯度、數字改變，以及原本凍結的數字維持不動。

前置是[8.8兩張小矩陣](https://birdhackor.github.io/tiny-perceptron-vlm/8.8.html)、[8.13縮放約定](https://birdhackor.github.io/tiny-perceptron-vlm/8.13.html)與[11.2更新器名單](https://birdhackor.github.io/tiny-perceptron-vlm/11.2.html)。原矩陣把輸入乘權重得到一份結果；LoRA讓同一輸入先經小矩陣A，再經小矩陣B，得到修正，加回原結果。A、B及縮放方式要連同指定模組保存，不能把任何小檔案配到任意底座。

下圖從上方的同一份輸入分成兩路：灰線用固定原矩陣W照常運算，藍線先經A縮小，再經B形成原矩陣需要的輸出尺寸。α是alpha，r是rank，縮放方框把這份修正乘上α/r；下方加號再把兩路結果相加。凍結原配方是固定其數字，不是把整條計算關掉。為了讓修正收到梯度，後續計算仍需保留求導路線。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural_base_adapter.svg")))

先在四項輸入、三項輸出的Linear旁真正更新一次：


In [ ]:
import torch
from torch import nn
from tiny_perceptron.alignment import LoRALinear

torch.manual_seed(42)
layer = LoRALinear(nn.Linear(4, 3), rank=2, alpha=2)
base_before = layer.base.weight.detach().clone()
branch_before = layer.b.detach().clone()
optimizer = torch.optim.SGD([p for p in layer.parameters() if p.requires_grad], lr=0.1)
loss = layer(torch.ones(1, 4)).square().mean()
loss.backward()
optimizer.step()
print("可更新數字", sum(p.numel() for p in layer.parameters() if p.requires_grad))
print("原矩陣完全未變", torch.equal(base_before, layer.base.weight))
print("修正B真的改變", not torch.equal(branch_before, layer.b))

`torch.ones(1, 4)`是一筆輸入，四項特徵都是1。`detach().clone()`把更新前的數值複製成獨立快照，不讓快照參與求導；若只記住同一份權重物件，更新後就無法拿舊值比較。本例的目標是讓三項輸出接近0，`square().mean()`取輸出平方的平均作代價；不是在教圖片或中文。A有2×4＝8個數，B有3×2＝6個數，所以第一行應印14，後兩行為True。優化器只收允許梯度的參數；凍結原矩陣後，它仍參與前向計算，但沒被這次更新修改。程式只有一輪新梯度，不需處理先前累積；若增加第二輪，先按[1.13](https://birdhackor.github.io/tiny-perceptron-vlm/1.13.html)清零。

實用底座的Linear通常很大，而且分佈在多個區塊。本課要明示把LoRA裝在哪些模組、rank是多少、合計多少可更新數字，並保留更新前後證據。這與全參訓練視覺入口不同：若視覺編碼器凍結，本課不能說它重新學過所有筆畫或所有物件。它提供已有視覺特徵，修正分支與共享語言模型學習如何在本任務使用它們。

只有少量數字更新，也不代表訓練能跳過大底座。回答的代價仍需經過它的計算回到分支，會有中間結果和時間成本。訓練後還要對照原底座的文字聊天、照片與中文題，保留改善、相同與退步；學習率、資料比例和參數範圍都要由實測選擇。

正式實驗把修正裝在語言模型28層注意力中的q與v線性投影；q負責形成查詢，v負責形成待混合的內容，直覺可回看[3.4](https://birdhackor.github.io/tiny-perceptron-vlm/3.4.html)。每處用rank 8、alpha 16。按照剛才的「A的數字＋B的數字」，可以從實際尺寸算出本輪更新規模：

| 每層的修正位置 | A的形狀 | B的形狀 | 可更新數字 |
| --- | --- | --- | ---: |
| q投影 | 8×2048 | 2048×8 | 32,768 |
| v投影 | 8×2048 | 1024×8 | 24,576 |

2048和1024是這份底座投影的實際尺寸。它讓多個查詢頭共用較少組內容，因此q和v的輸出格數不同；這種分組方式詳見[16.4](https://birdhackor.github.io/tiny-perceptron-vlm/16.4.html)。每層有32,768＋24,576＝57,344個修正參數，28層合計1,605,632個。每處保存A、B兩份數字表，每層兩處，所以共有28×2×2＝112份修正張量。圖文底座的2,127,532,032個參數與整個語音辨識器維持凍結，沒有訓練視覺入口的新權重。

題目、先前對話與圖片位置提供線索，這輪只用最後一段助手示範回答計算代價，包括正常結束的符號；不要求模型照抄整份輸入。[實際編碼核對](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/training-mask/result.json)逐筆檢查272筆訓練記錄，最長輸入578個文字或視覺位置，低於2048上限，沒有靜默裁切。這是輸入與標籤檢查，不是訓練成績。

L4上的[第一份微調紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/train/result.json)完成180次更新，每次合併兩道題的梯度，累計360次題目出現與5,791個有效回答位置；272筆訓練記錄都至少使用一次；其中包含上一節提到的重複聊天範例。學習率為0.0001、種子42。不同題的回答長度不同，所以兩題的代價按有效回答位置數加權，不能把較短和較長回答的平均直接等量相加。

更新前後，112份LoRA張量的內容指紋都改變；7份底座張量中事先選定的數值樣本保持相同。這裡只抽查部分底座數字，沒有逐一比對全部21億個值。更新器的名單也只包含LoRA參數；每一步代價與梯度都檢查為有限值。這些證據證明這輪真的更新了指定修正，是否更會回答仍要看留出題。

模型載入、訓練與迴圈內的檢查點存檔共149.47秒；計時在最後核對與最終存檔之前停止，也不含資料檔案核驗、容器啟動、映像建置或Hugging Face傳輸。PyTorch記錄的GPU已配置記憶體峰值為5,615,273,984 bytes，約5.23 GiB；它不包含所有驅動與非PyTorch占用，也不是保證所有圖片長度都能放進相同容量。這份180步候選從固定底座新建修正，並非接續第19章權重；需要續訓時，必須用另一份完整更新器與隨機狀態，不能只拿推論修正檔。

第一次留出驗證發現，微調改善了一些短回答，卻也讓五道照片描述陷入重複、用完回答長度。因此我們另做一次較輕的更新：從同一底座重新建立修正，仍跑180步，只把學習率從0.0001降成0.00003。學習率控制每次調整的幅度；改小是否有幫助，要由新回答判斷，不能只看訓練代價下降。

[兩次訓練的實際對照](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/train-gentle/controlled-comparison.json)確認112份初始修正的指紋、每步題目順序與有效回答位置數都相同。第二次也完成180步、360次題目出現，112份修正都更新，底座的七份取樣維持相同；依同一計時範圍，模型載入、訓練與迴圈內的檢查點存檔共155.47秒，已配置記憶體峰值同樣約5.23 GiB。兩次時間的少量差距不是效能優劣的證據。接著用相同留出題與回答長度比較，再選交付版本；[20.5](https://birdhackor.github.io/tiny-perceptron-vlm/20.5.html)會說明怎樣核對真正的回答。

練習只把rank=2改1，alpha仍2，先算可更新數字變成4＋3＝7，再執行核對。原矩陣仍應未變，修正仍可更新；但縮放也從2/2變2/1，所以不能只拿一次loss差異宣稱較低rank比較好。
